In [ ]:
import pandas as pd

In [ ]:
file_name = "사내SNS댓글_데이터.csv"    # 파일 변수로 저장
df = pd.read_csv(file_name)
print(df.shape)        # (200,8) 200개의 행과 8개의 열.
df.columns      # 데이터 전처리를 하기 위해 컬럼명 알아보기


(200, 8)


Index(['raw_row_id', 'post_id', 'comment_id', 'author', 'comment_text',
       'published_at', 'like_count', 'reply_count'],
      dtype='str')

In [ ]:
# 1. comment id 결측값 삭제하기 ---------------
df['comment_id'].isna().sum()   # 몇개의 행이 비어있는지 확인 : 5개
df = df.dropna(subset=['comment_id'])   # comment_id 열만 보고 값이 빈 행을 삭제해
df.shape          # 결측값 삭제 이후 5개의 행이 사라짐. (195,8)

(195, 8)

In [ ]:
# 2. 댓글 내용이 비어 있는 행 삭제하기 -----------------
df['comment_text'].isna().sum()   # 몇개의 행이 비어있는지 확인: 3개
df = df.dropna(subset=['comment_text'])   # comment_text 열에 빈 행 삭제해
df.shape           # 결측값 삭제 이후 3개의 행 사라짐. (192, 8)

(192, 8)

In [ ]:
# 3. comment_id가 중복된 경우 처음 행만 남긴다 --------------
df['comment_id'].duplicated().sum()    # 중복행 개수 확인 : 10개
df = df.drop_duplicates(subset=['comment_id'], keep='first')    # 중복행 삭제하고, first만 남긴다.
df.shape

(182, 8)

In [ ]:
# 4. 작성자가 비어있으면 알 수 없음 으로 입력한다 ----------------
df['author'].isna().sum()        # 빈 행 개수 확인: 5개
df['author'] = df['author'].fillna('알 수 없음')
df['author'].isna().sum()        # 0개 됨.

np.int64(0)

In [ ]:
# 5.  댓글 내용의 HTML 태그, 줄바꿈, 탭, 연속 공백을 정리한다 -------------------

df['comment_text'].head(10)  # 어떤 태그가 있는지 확인
df['comment_text'] = df['comment_text'].str.replace( r'<[^>]+>',   #  <로 시작해서 >로 끝나는 부분 찾아
                                                     ' ',          # 공백으로 바꿔라.
                                                     regex=True)   # regular expression -> 정규표현식을 쓴다는 뜻.
df['comment_text'] = df['comment_text'].str.replace( r'[\r\n\t]+', # 줄바꿈 계열 문자, 줄바꿈, 탭 찾아
                                                     ' ',          # 공백으로 바꿔라.
                                                     regex=True)
df['comment_text'] = df['comment_text'].str.replace( r' +',        #  여러 개의 공백을 찾아
                                                     ' ',          # 공백으로 바꿔라.
                                                     regex=True)
df['comment_text'] = df['comment_text'].str.strip()                # 앞, 뒤에 남은 공백 없애라.
df['comment_text'].head(10)                                        # 전처리 된 댓글 확인.

0        컨퍼런스 참가 지원 문의는 어디로 하면 되나요? ㅎㅎ
1    사내 셔틀버스 운행 관련 자료는 어디서 볼 수 있나요? ㅠㅠ
2            카풀 지원 제도 신청서는 어디서 받나요? ㅎㅎ
3               사내 자격증 지원 대상 조건이 궁금합니다
4               사내 설문조사 덕분에 도움 많이 됐습니다
5                           빨리 신청해야겠어요
6             이번에 저도 사내 메신저 개편 참여해보려고요
7            출장비 정산 방식 자세한 설명 부탁드려요 ㅠㅠ
8           타 부서도 사내 교육 프로그램 이용 가능한가요?
9               여름 휴가 정책 신청서는 어디서 받나요?
Name: comment_text, dtype: str

In [ ]:
# 6. like_count와 reply_count 에서 숫자만 추출하여 정수로 변환한다 ---------------

df[['like_count','reply_count']].head(15)  # 실제로 어떤 값이 들어있는지 확인하기.

df['like_count'] = df['like_count'].str.replace( r'[^0-9]',      # 숫자 제외한 문자는
                                                 '',             # 삭제해라
                                                 regex=True)
df['reply_count'] = df['reply_count'].str.replace( r'[^0-9]',      # 숫자 제외한 문자는
                                                 '',               # 삭제해라
                                                 regex=True)

df['like_count'] = df['like_count'].astype(int)                    # 정수로 변환해라.
df['reply_count'] = df['reply_count'].astype(int)

In [ ]:
df[['like_count','reply_count']].head(15)        # 다시 확인하면 숫자만 남음.
df[['like_count','reply_count']].dtypes        # 다시 확인하면 숫자만 남음.

like_count     int64
reply_count    int64
dtype: object

In [ ]:
# 7. published_at을 날짜형으로 변환하고 변환되지 않는 행을 삭제한다----------------

df['published_at'] = pd.to_datetime( df['published_at'],    # 날짜형으로 바꾼다.
                                     format='mixed',
                                     errors='coerce')       # 날짜로 바꿀 수 없는 값은 NaT로 바꿔

df = df.dropna(subset=['published_at'])      # 날짜형에 빈칸은 삭제한다.
# df['published_at'].dtype           # dtype('<M8[us]') 로 나왔기 때문에, 날짜형임을 확인.
df.shape

(177, 8)

In [ ]:
# 8. 전처리 완료 후 파일 저장 시 utf-8-sig 형식으로 인코딩한다. -------------------

df.to_csv('사내SNS댓글_전처리완료.csv',
          index=False,
          encoding='utf-8-sig')

In [ ]:
# csv로 저장됐는지 확인.
import os
os.path.exists('사내SNS댓글_전처리완료.csv')


True

In [ ]:
# 저장된 csv 파일을 메일로 전송하기.- -- -------------------------------
import smtplib
from email.message import EmailMessage
from getpass import getpass

sender_email = input("발신 이메일 주소: ").strip()     # 보내는 메일
receiver_email = input("수신 이메일 주소: ").strip()   # 받는 메일

file_name = '사내SNS댓글_전처리완료.csv'     # 첨부 파일

app_password = getpass("Gmail 앱 비번을 입력해주세요")   # 지메일 앱 비번

msg = EmailMessage()

msg["Subject"] = "[댓글수집] 사내 SNS 댓글 전처리 결과"    # 메일 제목
msg["From"] = sender_email
msg["To"] = receiver_email

msg.set_content("전처리가 완료된 댓글 CSV 파일을 첨부합니다.")

with open(file_name, "rb") as f:
    file_data = f.read()

# 이 첨부파일은 csv파일이고, 이걸 이메일에 첨부파일로 넣어줘!
msg.add_attachment( file_data,
                    maintype="text",
                    subtype='csv',
                    filename=file_name)

# Gmail 서버에 접속해서 진짜 이메일 보내기
with smtplib.SMTP("smtp.gmail.com", 587) as smtp:
    smtp.starttls()                # 통신 암호화
    smtp.login(sender_email, app_password)      #내 지메일 계정에 로그인
    smtp.send_message(msg)

print("메일 전송 완료")


앱 비번을 입력해주세요 ········


메일 전송 완료


In [ ]:
file_name = "사내SNS댓글_전처리완료.csv"    # 파일 변수로 저장
df = pd.read_csv(file_name)
df.shape

(111, 8)

<StringDtype(storage='python', na_value=nan)>